# Prueba local del modelo XGBoost registrado en MLflow

Este notebook prepara una fila de características tabulares y carga el modelo XGBoost del run más reciente que contiene `best-xgboost-model`. El endpoint de la API se conectará posteriormente.

In [11]:
from pathlib import Path
import os
import sys
import mlflow
import joblib
import numpy as np
import pandas as pd
import mne
import mlflow

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data.loader import load_subject, find_subject_pairs
from src.data.segmentation import extract_epochs
from src.preprocessing.feature_engineering import extract_features

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://32.194.151.154:5000')
MLFLOW_EXPERIMENT_NAMES = tuple(name.strip() for name in os.getenv(
    'MLFLOW_XGBOOST_EXPERIMENT_NAMES', 'sleep-staging-baseline-multicanal'
).split(',') if name.strip())
MLFLOW_ARTIFACT_PATH = os.getenv('MLFLOW_XGBOOST_ARTIFACT_PATH', 'best-xgboost-multicanal.joblib')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print('Proyecto:', PROJECT_ROOT)
print('MLflow:', MLFLOW_TRACKING_URI)
print('Experimentos:', MLFLOW_EXPERIMENT_NAMES)

Proyecto: C:\Users\danos\Documents\MAIA\Cursos\Semestre 4\Proyecto DS\proyecto-desarrollo-de-soluciones-G5
MLflow: http://32.194.151.154:5000
Experimentos: ('sleep-staging-baseline-multicanal',)


In [12]:
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient

def find_xgboost_run_id(artifact_path=MLFLOW_ARTIFACT_PATH):
    client = MlflowClient()
    experiments = [e for e in client.search_experiments(view_type=ViewType.ACTIVE_ONLY)
                   if e.name in MLFLOW_EXPERIMENT_NAMES]
    if not experiments:
        raise LookupError(f'No se encontraron: {MLFLOW_EXPERIMENT_NAMES}')

    runs = []
    for experiment in experiments:
        runs.extend(client.search_runs(
            [experiment.experiment_id],
            filter_string="attributes.status = 'FINISHED'",
            max_results=50000,
        ))

    for run in sorted(runs, key=lambda r: r.info.start_time or 0, reverse=True):
        items = client.list_artifacts(run.info.run_id)
        if any(not item.is_dir and item.path == artifact_path for item in items):
            return run.info.run_id
    raise FileNotFoundError(f'No se encontró el modelo {artifact_path}')

RUN_ID = find_xgboost_run_id()
MODEL_URI = f'runs:/{RUN_ID}/{MLFLOW_ARTIFACT_PATH}'
print('Run seleccionado:', RUN_ID)
print('Model URI:', MODEL_URI)

Run seleccionado: 4a7712d0b58548d4a4134f059c2e4bc6
Model URI: runs:/4a7712d0b58548d4a4134f059c2e4bc6/best-xgboost-multicanal.joblib


In [13]:
# Celda intermedia conservada del notebook original; no ejecutar.

In [14]:
# Descarga y carga el archivo joblib registrado como artefacto MLflow.
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path=MLFLOW_ARTIFACT_PATH,
)
loaded_artifact = joblib.load(artifact_path)
print('Tipo de artefacto:', type(loaded_artifact).__name__)
if isinstance(loaded_artifact, dict):
    print('Claves del artefacto:', list(loaded_artifact))
    model = loaded_artifact.get('model')
    model_feature_columns = loaded_artifact.get('feature_columns')
    label_encoder = loaded_artifact.get('label_encoder')
    class_labels = loaded_artifact.get('class_labels')
else:
    model = loaded_artifact
    model_feature_columns = None
    label_encoder = None
    class_labels = ['W', 'N1', 'N2', 'N3', 'REM']
if model is None or not hasattr(model, 'predict'):
    raise TypeError('El artefacto no contiene un modelo con método predict.')
print('Modelo:', type(model).__name__)
print('Número de características esperadas:', model.n_features_in_)

Tipo de artefacto: dict
Claves del artefacto: ['model', 'label_encoder', 'feature_columns', 'excluded_columns', 'categorical_columns', 'class_labels']
Modelo: XGBClassifier
Número de características esperadas: 49


C:\Users\danos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
# Convierte una señal EDF directamente en las características del modelo.
data_dir = PROJECT_ROOT / 'data' / 'sleep-edfx' / 'sleep-cassette'
pairs = find_subject_pairs(data_dir)
if not pairs:
    raise FileNotFoundError(f'No se encontraron pares PSG/Hypnograma en {data_dir}')
psg_path, hypno_path = pairs[0]
subject = load_subject(psg_path, hypno_path)
available = subject.raw.ch_names
channel_picks = [next((ch for ch in available if 'Fpz-Cz' in ch or 'FPZ-CZ' in ch.upper()), None), next((ch for ch in available if 'Pz-Oz' in ch or 'PZ-OZ' in ch.upper()), None), next((ch for ch in available if 'EOG' in ch.upper()), None)]
channel_picks = [ch for ch in channel_picks if ch is not None]
epoch_ds = extract_epochs(subject.raw, subject.annotations, recording_id=subject.subject_id, channel_picks=channel_picks)
feature_ds = extract_features(epoch_ds)
features_df = pd.DataFrame(feature_ds.X, columns=feature_ds.feature_names)
missing = [c for c in model_feature_columns if c not in features_df.columns]
if missing:
    print('Advertencia: características no generadas:', missing)
X = features_df.reindex(columns=model_feature_columns, fill_value=0.0)
X_one = X
predicted_encoded = model.predict(X)
probabilities = model.predict_proba(X)
if label_encoder is not None:
    predicted_labels = label_encoder.inverse_transform(predicted_encoded.astype(int))
else:
    predicted_labels = predicted_encoded
confidence = probabilities.max(axis=1)
prob_df = pd.DataFrame(probabilities, columns=class_labels)
prob_df.insert(0, 'epoch', np.arange(1, len(prob_df) + 1))
prob_df['prediction'] = predicted_labels
prob_df['confidence'] = confidence
print('Características enviadas:', X_one.shape)
print('Características generadas:', X.shape)
prob_df

C:\Users\danos\Documents\MAIA\Cursos\Semestre 4\Proyecto DS\proyecto-desarrollo-de-soluciones-G5\src\data\loader.py:38: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  return mne.io.read_raw_edf(path, preload=False, verbose=verbose)
C:\Users\danos\Documents\MAIA\Cursos\Semestre 4\Proyecto DS\proyecto-desarrollo-de-soluciones-G5\src\data\loader.py:38: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  return mne.io.read_raw_edf(path, preload=False, verbose=verbose)
C:\Users\danos\Documents\MAIA\Cursos\Semestre 4\Proyecto DS\proyecto-desarrollo-de-soluciones-G5\src\data\loader.py:38: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  return mne.io.read_raw_edf(path, preload=False, verbose=verbose)


[loader] Aviso: el hypnograma termina en 86400.0s pero el PSG dura 79500.0s. Probablemente es la cola '?' de relleno; se ignorará al extraer épocas (C:\Users\danos\Documents\MAIA\Cursos\Semestre 4\Proyecto DS\proyecto-desarrollo-de-soluciones-G5\data\sleep-edfx\sleep-cassette\SC4001E0-PSG.edf).
Advertencia: características no generadas: ['channel_count', 'has_eeg_pz_oz', 'has_eog_horizontal', 'channel_combo_eeg_fpz_cz', 'channel_combo_eeg_fpz_cz__eeg_pz_oz', 'channel_combo_eeg_fpz_cz__eeg_pz_oz__eog_horizontal', 'channel_combo_eeg_fpz_cz__eog_horizontal']
Características enviadas: (2650, 49)
Características generadas: (2650, 49)


,epoch,W,N1,N2,N3,REM,prediction,confidence
0,1,0.000484,0.000470,0.000205,0.007097,0.991744,W,0.991744
1,2,0.000498,0.000199,0.000054,0.000071,0.999177,W,0.999177
2,3,0.222601,0.005128,0.017107,0.006021,0.749143,W,0.749143
3,4,0.120409,0.004147,0.003079,0.008092,0.864273,W,0.864273
4,5,0.103456,0.015254,0.000668,0.080988,0.799634,W,0.799634
...,...,...,...,...,...,...,...,...
2645,2646,0.010130,0.001640,0.000161,0.010178,0.977890,W,0.977890
2646,2647,0.001202,0.000568,0.000222,0.000375,0.997633,W,0.997633
2647,2648,0.013475,0.003935,0.000678,0.064232,0.917681,W,0.917681
2648,2649,0.001898,0.000695,0.000647,0.000685,0.996075,W,0.996075


In [18]:
import requests
import pandas as pd

response = requests.post(
    "http://localhost:8000/predict-xgboost",
    json={
        "channel_names": channel_picks,
        "sampling_rate_hz": 100,
        "signal_unit": "uV",
        "model_artifact": "best-xgboost-multicanal.joblib",
        "epochs": epoch_ds.X.tolist(),
    },
)

response.raise_for_status()

api_result = response.json()

prob_df = pd.DataFrame(api_result["predictions"])

prob_df

,epoch_index,predicted_stage,confidence,probabilities
0,0,W,0.990525,"{'W': 0.0005555436946451664, 'N1': 0.000539339..."
1,1,W,0.999177,"{'W': 0.0004984517581760883, 'N1': 0.000199067..."
2,2,W,0.728038,"{'W': 0.24132858216762543, 'N1': 0.00555927352..."
3,3,W,0.852556,"{'W': 0.13080355525016785, 'N1': 0.00450491765..."
4,4,W,0.784679,"{'W': 0.11117786914110184, 'N1': 0.01639216206..."
...,...,...,...,...
2645,2645,W,0.976948,"{'W': 0.010562228038907051, 'N1': 0.0017102505..."
2646,2646,W,0.997327,"{'W': 0.0013568622525781393, 'N1': 0.000641739..."
2647,2647,W,0.914396,"{'W': 0.014012675732374191, 'N1': 0.0040919417..."
2648,2648,W,0.995664,"{'W': 0.002097425749525428, 'N1': 0.0007676620..."
